#  Plume Care Navigator Synthetic Data Generator

**Portfolio project for:** Senior Data Engineer (Data + Applied AI) at Plume Clinic

This notebook generates the full **2,680,000-patient** synthetic dataset representing Plume's estimated addressable market:
- US Population (2024 Census): **335,000,000**
- Trans prevalence (~1%): **3,350,000**
- Plume market share (~80%): **2,680,000 patients**

### What this generates
| Table | Rows (approx) | Description |
|---|---|---|
| `patients` | 2,680,000 | Demographics, HRT regimen, state, start date |
| `subscriptions` | 2,680,000 | Stripe billing data — plan, MRR, churn status |
| `lab_results` | ~80,000,000 | Quarterly hormone + metabolic lab panels |
| `clinical_visits` | ~19,600,000 | Healthie appointment records |

### Runtime recommendation
> **Runtime → Change runtime type → RAM: High-RAM, GPU: None**  
> The lab results table (~80M rows) requires ~12 GB of RAM. A standard Colab runtime (12 GB) may be tight; High-RAM (52 GB) is recommended.

### Output options
- **Option A:** Download Parquet files directly to your machine
- **Option B:** Upload directly to Google BigQuery (requires GCP project)


## 1. Setup & Configuration

In [ ]:
# Install required packages (pyarrow is pre-installed in Colab, but ensure latest)
!pip install pyarrow --quiet
print('Dependencies are ready')

In [ ]:
import os
import uuid
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Change N_PATIENTS to 1_000_000 for a quick test run on standard Colab
N_PATIENTS  = 2_680_000   # Full addressable market
N_PROVIDERS = 500
CHUNK_SIZE  = 200_000     # Increase if using High-RAM runtime
RANDOM_SEED = 42
OUTPUT_DIR  = '/content/plume_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

rng = np.random.default_rng(RANDOM_SEED)

print(f'Configuration:')
print(f'  N_PATIENTS  = {N_PATIENTS:,}')
print(f'  CHUNK_SIZE  = {CHUNK_SIZE:,}')
print(f'  OUTPUT_DIR  = {OUTPUT_DIR}')

## 2. Lookup Tables & Helper Functions

In [ ]:
STATES = [
    'CA','TX','FL','NY','PA','IL','OH','GA','NC','MI',
    'NJ','VA','WA','AZ','MA','TN','IN','MO','MD','WI',
    'CO','MN','SC','AL','LA','KY','OR','OK','CT','UT',
    'IA','NV','AR','MS','KS','NM','NE','WV','ID','HI',
    'NH','ME','MT','RI','DE','SD','ND','AK','VT','WY',
]
STATE_WEIGHTS = np.array([
    39.5,30.0,22.6,19.6,13.0,12.6,11.8,11.0,10.7,10.0,
     9.3, 8.7, 7.8, 7.4, 7.0, 7.0, 6.8, 6.2, 6.2, 5.9,
     5.8, 5.7, 5.3, 5.1, 4.6, 4.5, 4.2, 4.0, 3.6, 3.4,
     3.2, 3.2, 3.1, 3.0, 2.9, 2.1, 2.0, 1.8, 1.9, 1.4,
     1.4, 1.4, 1.1, 1.1, 1.0, 0.9, 0.8, 0.7, 0.6, 0.6,
])
STATE_WEIGHTS /= STATE_WEIGHTS.sum()

GENDER_IDENTITIES = [
    'Trans Woman','Trans Man','Non-binary','Genderqueer',
    'Gender Non-conforming','Agender','Two-Spirit','Other',
]
GENDER_WEIGHTS = np.array([0.30,0.28,0.22,0.08,0.06,0.03,0.01,0.02])

HRT_MAP = {
    'Trans Woman': (
        ['Estradiol + Spironolactone','Estradiol + Progesterone','Estradiol Monotherapy'],
        [0.55, 0.30, 0.15],
    ),
    'Trans Man': (
        ['Testosterone (Injection)','Testosterone (Gel/Patch)'],
        [0.65, 0.35],
    ),
    'default': (
        ['Estradiol Monotherapy','Testosterone (Gel/Patch)','No HRT'],
        [0.30, 0.30, 0.40],
    ),
}

LAB_TESTS = {
    'Estradiol':    ('pg/mL',  150.0, 40.0,  50.0, 300.0),
    'Testosterone': ('ng/dL',  500.0,120.0,  30.0, 900.0),
    'Hemoglobin':   ('g/dL',    13.5,  1.2,  11.5,  17.5),
    'Potassium':    ('mmol/L',   4.2,  0.4,   3.5,   5.5),
}

def assign_hrt_vectorized(genders):
    result = np.empty(len(genders), dtype=object)
    for g, (options, probs) in HRT_MAP.items():
        mask = genders == g
        if mask.any():
            result[mask] = rng.choice(options, size=mask.sum(), p=probs)
    remaining = result == None
    if remaining.any():
        options, probs = HRT_MAP['default']
        result[remaining] = rng.choice(options, size=remaining.sum(), p=probs)
    return result

def lab_params(hrt_arr, test):
    is_e = np.array(['Estradiol' in str(h) for h in hrt_arr])
    is_t = np.array(['Testosterone' in str(h) for h in hrt_arr])
    n = len(hrt_arr)
    if test == 'Estradiol':
        return np.where(is_e, 150.0, 25.0), np.where(is_e, 40.0, 10.0)
    elif test == 'Testosterone':
        return (
            np.where(is_t, 500.0, np.where(is_e, 35.0, 300.0)),
            np.where(is_t, 120.0, np.where(is_e, 15.0,  80.0)),
        )
    elif test == 'Hemoglobin':
        return np.where(is_t, 15.5, 13.0), np.full(n, 1.2)
    else:
        return np.full(n, 4.2), np.full(n, 0.4)

print('Lookup tables and helpers defined')

## 3. Generate All Tables

In [ ]:
print('[1/4] Generating patients ...')

patient_ids     = np.array([str(uuid.uuid4()) for _ in range(N_PATIENTS)])
patient_states  = rng.choice(STATES, size=N_PATIENTS, p=STATE_WEIGHTS)
patient_genders = rng.choice(GENDER_IDENTITIES, size=N_PATIENTS, p=GENDER_WEIGHTS)
patient_hrt     = assign_hrt_vectorized(patient_genders)

today           = pd.Timestamp('2024-12-31')
age_days        = rng.integers(18*365, 65*365, size=N_PATIENTS)
birth_dates     = (today - pd.to_timedelta(age_days, unit='D')).date
start_offsets   = rng.integers(0, 730, size=N_PATIENTS)
start_dates_ts  = pd.Timestamp('2023-01-01') + pd.to_timedelta(start_offsets, unit='D')
start_dates     = start_dates_ts.date
is_active       = rng.random(N_PATIENTS) > 0.12

patients_df = pd.DataFrame({
    'patient_id':      patient_ids,
    'state':           patient_states,
    'gender_identity': patient_genders,
    'hrt_regimen':     patient_hrt,
    'birth_date':      birth_dates,
    'start_date':      start_dates,
    'is_active':       is_active,
})
out = os.path.join(OUTPUT_DIR, 'patients.parquet')
patients_df.to_parquet(out, index=False, compression='snappy')
print(f'{len(patients_df):,} rows | {os.path.getsize(out)/1e6:.1f} MB')

In [ ]:
print('[2/4] Generating subscriptions ...')

plan_types  = rng.choice(['Monthly','Annual'], size=N_PATIENTS, p=[0.70,0.30])
mrr         = np.where(plan_types == 'Monthly', 99.0, 82.50)
statuses    = np.where(
    is_active,
    rng.choice(['Active','Past Due'], size=N_PATIENTS, p=[0.95,0.05]),
    'Canceled',
)
end_offsets   = rng.integers(30, 730, size=N_PATIENTS)
end_dates_raw = (start_dates_ts + pd.to_timedelta(end_offsets, unit='D')).date
end_dates     = np.where(statuses == 'Canceled', end_dates_raw, None)

subs_df = pd.DataFrame({
    'subscription_id': [str(uuid.uuid4()) for _ in range(N_PATIENTS)],
    'patient_id':      patient_ids,
    'plan_type':       plan_types,
    'status':          statuses,
    'mrr':             mrr,
    'start_date':      start_dates,
    'end_date':        end_dates,
})
out = os.path.join(OUTPUT_DIR, 'subscriptions.parquet')
subs_df.to_parquet(out, index=False, compression='snappy')
print(f'  {len(subs_df):,} rows | {os.path.getsize(out)/1e6:.1f} MB')

In [ ]:
print('[3/4] Generating lab_results (chunked — this takes ~10 min on High-RAM) ...')

lab_schema = pa.schema([
    pa.field('lab_id',       pa.string()),
    pa.field('patient_id',   pa.string()),
    pa.field('date',         pa.date32()),
    pa.field('test_name',    pa.string()),
    pa.field('result_value', pa.float32()),
    pa.field('unit',         pa.string()),
    pa.field('flag',         pa.string()),
])

start_dates_np = pd.to_datetime(patients_df['start_date']).values
lab_writer     = pq.ParquetWriter(os.path.join(OUTPUT_DIR, 'lab_results.parquet'), lab_schema, compression='snappy')
total_labs     = 0

for cs in range(0, N_PATIENTS, CHUNK_SIZE):
    ce      = min(cs + CHUNK_SIZE, N_PATIENTS)
    n       = ce - cs
    c_ids   = patient_ids[cs:ce]
    c_hrt   = patient_hrt[cs:ce]
    c_start = start_dates_np[cs:ce]
    c_act   = is_active[cs:ce]

    n_panels = np.where(c_act, 8, rng.integers(1, 8, size=n))
    rep_ids  = np.repeat(c_ids,   n_panels)
    rep_hrt  = np.repeat(c_hrt,   n_panels)
    rep_st   = np.repeat(c_start, n_panels)
    p_idx    = np.concatenate([np.arange(k) for k in n_panels])
    lab_dt   = (rep_st + (p_idx * 91).astype('timedelta64[D]')).astype('datetime64[D]')
    nr       = len(rep_ids)

    frames = []
    for test, (unit, _, _, low, high) in LAB_TESTS.items():
        means, stds = lab_params(rep_hrt, test)
        vals  = np.maximum(0.0, rng.normal(means, stds)).astype(np.float32)
        flags = np.where(vals < low, 'Low', np.where(vals > high, 'High', 'Normal'))
        frames.append(pd.DataFrame({
            'lab_id':       [str(uuid.uuid4()) for _ in range(nr)],
            'patient_id':   rep_ids,
            'date':         lab_dt,
            'test_name':    test,
            'result_value': vals,
            'unit':         unit,
            'flag':         flags,
        }))

    chunk_df = pd.concat(frames, ignore_index=True)
    chunk_df['date'] = pd.to_datetime(chunk_df['date']).dt.date
    lab_writer.write_table(pa.Table.from_pandas(chunk_df, schema=lab_schema, preserve_index=False))
    total_labs += len(chunk_df)
    print(f'  {ce:>9,} / {N_PATIENTS:,} patients | {total_labs:,} lab rows')

lab_writer.close()
print(f'  {total_labs:,} rows | {os.path.getsize(os.path.join(OUTPUT_DIR, "lab_results.parquet"))/1e6:.1f} MB')

In [ ]:
print('[4/4] Generating clinical_visits (chunked) ...')

VISIT_TYPES    = ['Intake','Follow-up HRT','Mental Health','Urgent','Lab Review']
VISIT_WEIGHTS  = np.array([0.10, 0.50, 0.20, 0.05, 0.15])
VISIT_STATUSES = ['Completed','No-Show','Canceled']
STATUS_WEIGHTS = np.array([0.85, 0.08, 0.07])
provider_ids   = np.array([str(uuid.uuid4()) for _ in range(N_PROVIDERS)])

visit_schema = pa.schema([
    pa.field('visit_id',    pa.string()),
    pa.field('patient_id',  pa.string()),
    pa.field('provider_id', pa.string()),
    pa.field('date',        pa.date32()),
    pa.field('visit_type',  pa.string()),
    pa.field('status',      pa.string()),
])

visit_writer = pq.ParquetWriter(os.path.join(OUTPUT_DIR, 'clinical_visits.parquet'), visit_schema, compression='snappy')
total_visits = 0

for cs in range(0, N_PATIENTS, CHUNK_SIZE):
    ce      = min(cs + CHUNK_SIZE, N_PATIENTS)
    n       = ce - cs
    c_ids   = patient_ids[cs:ce]
    c_start = start_dates_np[cs:ce]
    c_act   = is_active[cs:ce]

    vpp     = np.where(c_act, rng.integers(6,11,size=n), rng.integers(1,5,size=n))
    rep_ids = np.repeat(c_ids,   vpp)
    rep_st  = np.repeat(c_start, vpp)
    nr      = len(rep_ids)

    offsets  = rng.integers(0, 730, size=nr)
    visit_dt = (rep_st + offsets.astype('timedelta64[D]')).astype('datetime64[D]')

    chunk_df = pd.DataFrame({
        'visit_id':    [str(uuid.uuid4()) for _ in range(nr)],
        'patient_id':  rep_ids,
        'provider_id': rng.choice(provider_ids, size=nr),
        'date':        pd.DatetimeIndex(visit_dt).date,
        'visit_type':  rng.choice(VISIT_TYPES,    size=nr, p=VISIT_WEIGHTS),
        'status':      rng.choice(VISIT_STATUSES, size=nr, p=STATUS_WEIGHTS),
    })
    visit_writer.write_table(pa.Table.from_pandas(chunk_df, schema=visit_schema, preserve_index=False))
    total_visits += nr
    print(f'  {ce:>9,} / {N_PATIENTS:,} patients | {total_visits:,} visit rows')

visit_writer.close()
print(f'   {total_visits:,} rows | {os.path.getsize(os.path.join(OUTPUT_DIR, "clinical_visits.parquet"))/1e6:.1f} MB')

## 4. Summary & Output Options

In [ ]:
files = ['patients.parquet','subscriptions.parquet','lab_results.parquet','clinical_visits.parquet']
total_size = sum(os.path.getsize(os.path.join(OUTPUT_DIR, f)) for f in files)
print('=' * 60)
print('  Generation complete — final summary')
print('=' * 60)
for f in files:
    meta = pq.read_metadata(os.path.join(OUTPUT_DIR, f))
    sz   = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f'  {f:<38}  {meta.num_rows:>12,} rows  {sz:>7.1f} MB')
print(f'  {""*57}')
print(f'  {"TOTAL":38}  {"":>12}       {total_size/1e6:>7.1f} MB')
print('=' * 60)

## 5a. Option A — Download files to your local machine
Run the cell below to zip and download all four Parquet files.

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/plume_synthetic_data.zip'
shutil.make_archive('/content/plume_synthetic_data', 'zip', OUTPUT_DIR)
print(f'Archive size: {os.path.getsize(zip_path)/1e6:.1f} MB')
files.download(zip_path)

## 5b. Option B — Upload directly to Google BigQuery

**Prerequisites:**
1. A Google Cloud project with BigQuery API enabled
2. A BigQuery dataset already created (e.g., `plume_bronze`)

Replace `YOUR_PROJECT_ID` and `YOUR_DATASET` below, then run the cell.

In [ ]:
from google.colab import auth
from google.cloud import bigquery

# Authenticate with GCP
auth.authenticate_user()

PROJECT_ID = 'YOUR_PROJECT_ID'   # e.g. 'plume-care-navigator'
DATASET    = 'plume_bronze'      # Must already exist in BigQuery

client = bigquery.Client(project=PROJECT_ID)

table_map = {
    'patients':         'patients.parquet',
    'subscriptions':    'subscriptions.parquet',
    'lab_results':      'lab_results.parquet',
    'clinical_visits':  'clinical_visits.parquet',
}

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    autodetect=True,
)

for table_name, filename in table_map.items():
    table_id  = f'{PROJECT_ID}.{DATASET}.{table_name}'
    file_path = os.path.join(OUTPUT_DIR, filename)
    print(f'Uploading {filename} → {table_id} ...')
    with open(file_path, 'rb') as f:
        job = client.load_table_from_file(f, table_id, job_config=job_config)
    job.result()  # Wait for completion
    tbl = client.get_table(table_id)
    print(f'  ✓ {tbl.num_rows:,} rows loaded into {table_id}')

print('\nAll tables loaded into BigQuery successfully!')